# Workshop 4 — learning representations from handwritten digits


We use 28 × 28 greyscale images of handwritten digits. Each image is a vector of 784
dimensionless intensity measurements, scaled between 0 and 1. An autoencoder learns a compressed
set of coordinates from which it tries to reconstruct those measurements.

**Start in Section 1.0.** Run the supplied Section 0 once using **Run All Above**, then leave it folded.
For each exercise, predict the relevant shape or behaviour, write your implementation, inspect the
result, and explain it in words. Complete each `## FINISH_ME ##` marker before running its cell.
Replace the marker with your own code; some markers stand for several lines. The supplied
mathematics and explanations support your reasoning.

Use the chevron beside a Markdown heading to collapse a section. In VS Code it appears when you hover
beside the heading. The notebook saves Section 0 in its collapsed state.

| Section | Main question | Your task |
|---|---|---|
| 1. Data handling | How does an image on disk become a batch of tensors? | Build the DataFrame → Dataset → DataLoader workflow. |
| 2. Dense AE | Can a shorter vector retain enough information to rebuild an image? | Assemble the model and follow a gradient update. |
| 3. VAE | How do we learn a distribution of latent coordinates? | Build the VAE around the supplied Gaussian mathematics. |
| 4. Latent space | What structure has each encoder learned? | Compare all digit classes in training and validation. |
| 5. Clustering | Can an additional objective organise the learned coordinates? | Implement and interpret unsupervised refinement. |
| 6. Anomaly investigation | Does the plain dense AE recognise an unfamiliar image? | Design and implement the guided investigation yourself. |

The input labels describe categories; the numerical distance between digit labels has no meaning.
Keep three ideas separate throughout: the training objective, a measurement made with a fitted model,
and a conclusion supported by that measurement.

**Section 6 is different:** it contains guidance and equations only. Create your own analysis cells,
figures and explanations using the experimental protocol below.


## 0. Technical setup — supplied


This folded section supplies imports, random seeds, file paths and the binary MNIST reader.
You can use that reader without implementing the binary file format yourself.

Sections 1–5 use PyTorch, torchvision, NumPy, Pandas, Matplotlib, scikit-learn and IPython. Custom model and plotting code lives inside this notebook. Missing MNIST data
is downloaded into the data folder, then read from disk into arrays.

Training lengths and the number of latent coordinates are defined in the settings cell below.
After changing a model definition or those settings, rerun the dependent cells so the fitted models
and their plots describe the same experiment. The image for Section 6 is introduced there.


In [ ]:
# Set up the numerical tools and the settings shared by the experiments.

import os
import random
import warnings
from pathlib import Path

# Hide only the recurring scikit-learn/BLAS matmul RuntimeWarnings.
# Other warnings, including non-matmul numerical warnings, remain visible.
warnings.filterwarnings(
    "ignore",
    message=r"^(divide by zero|overflow|invalid value) encountered in matmul$",
    category=RuntimeWarning,
    module=r"^sklearn\.",
)

THREADS = 10  # Number of CPU threads available to the tensor calculations.
for variable in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS"):
    os.environ.setdefault(variable, str(THREADS))

import copy  # Copy the fitted AE before refining it for clustering.
import numpy as np
import pandas as pd
import torch
from torch import nn                                    # layers with weights: nn.Linear, nn.Embedding, ...
from torch.nn import functional as F                    # functions without weights: losses, one_hot, ...
from torch.utils.data import Dataset, DataLoader, TensorDataset  # dataset access and batching
from torchvision.datasets import MNIST

# Download data from the dataset maintainers over HTTPS; torchvision verifies the file checksums.
MNIST.mirrors = ["https://ossci-datasets.s3.amazonaws.com/mnist/"]
CLASS_NAMES = ("0", "1", "2", "3", "4", "5", "6", "7", "8", "9")
NUM_CLASSES = len(CLASS_NAMES)
import matplotlib.pyplot as plt

# ---- Settings shared by every model ----
SEED = 12345             # repeatable splits and initialisation within the same software environment
BATCH_SIZE = 256         # images per gradient step: 50,000 / 256 ≈ 196 steps per epoch
LATENT_DIM = 10           # Number of learned coordinates per image; figures display selected pairs.
AE_EPOCHS = 80           # one epoch = one pass over all 50,000 training images
VAE_EPOCHS = 80
DEC_EPOCHS = 160          # refinement epochs, after the AE's own training
torch.manual_seed(SEED)  # seeds PyTorch's global random generator (initial weights, VAE noise)
torch.set_num_threads(THREADS)
random.seed(SEED)
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

from IPython.display import Markdown, display


In [ ]:
# Locate the MNIST files; download the data only if the local copies are missing.

DATA_ROOT = Path("data")
raw_directory = DATA_ROOT / "MNIST" / "raw"
disk_files = {
    "train_images": raw_directory / "train-images-idx3-ubyte",
    "train_labels": raw_directory / "train-labels-idx1-ubyte",
    "test_images": raw_directory / "t10k-images-idx3-ubyte",
    "test_labels": raw_directory / "t10k-labels-idx1-ubyte",
}
# Preparation only: torchvision downloads and extracts any missing data files.
# The DataFrame below is constructed by reading the files on disk, not .data tensors.
if not all(path.is_file() for path in disk_files.values()):
    MNIST(DATA_ROOT, train=True, download=True)
    MNIST(DATA_ROOT, train=False, download=True)
for name, path in disk_files.items():
    print(f"{name}: {path} ({path.stat().st_size:,} bytes)")


In [ ]:
# Supplied binary reader: separate the file headers from the pixel and label arrays.

import struct

def read_mnist_idx(image_path, label_path):
    """Read an on-disk IDX image/label pair. Binary-format parsing is supplied."""
    image_bytes = Path(image_path).read_bytes()
    label_bytes = Path(label_path).read_bytes()
    # The first bytes describe the file; the remaining bytes are the measurements.
    image_magic, image_count, height, width = struct.unpack(">IIII", image_bytes[:16])
    label_magic, label_count = struct.unpack(">II", label_bytes[:8])
    if (image_magic, label_magic) != (2051, 2049):
        raise ValueError("Expected IDX image and label files, in that order.")
    if image_count != label_count or (height, width) != (28, 28):
        raise ValueError("Images and labels must match and every image must be 28 × 28.")
    if len(image_bytes) != 16 + image_count * height * width or len(label_bytes) != 8 + label_count:
        raise ValueError("IDX file size does not match its header; check the downloaded files.")
    # View the byte values as unsigned intensities, then group 784 values per image.
    pixels = np.frombuffer(image_bytes, dtype=np.uint8, offset=16).reshape(image_count, 784)
    labels = np.frombuffer(label_bytes, dtype=np.uint8, offset=8)
    return pixels, labels


## 1.0 Data handling


An image becomes a row of 784 pixel measurements and one separate category label.
MNIST provides 60,000 training images and 10,000 test images. We split the first set into
50,000 images for fitting and 10,000 for validation, using the same split for every model.

**Follow the data:** files on disk → arrays → Pandas tables → a Dataset → batches from a DataLoader.

| Object | Shape | Meaning of each axis |
|---|---|---|
| Pixel table | 784 pixel columns, plus a label column | One row per image |
| Image tensor | $(N,1,28,28)$ | Images, colour channels, rows, columns |
| Label tensor | $N$ | One digit category per image |
| Latent tensor | $(N,d)$ | One vector of $d$ learned coordinates per image |

The pixel values are divided by 255 when the table is converted to tensors. The plain AE uses
each image as both input and reconstruction target. Labels are retained for later grouping and plots;
none of the models uses digit labels as training inputs or targets.

Treat the official test set as a final independent measurement, rather than another source of
training or tuning examples.


### 1.1 Load the disk data into a Pandas DataFrame


The supplied reader returns two arrays: 784 pixel values per image and one category per image.
Your task starts with those arrays. Inspect a few rows and the table dimensions before moving on.
The label column is metadata about an image, not an input pixel.


Create and inspect the DataFrames:

1. Create each table with `pd.DataFrame(pixel_array, columns=PIXEL_COLUMNS)`.
2. Add its labels using `frame["label"] = label_array`.
3. Use `.shape` to inspect the row and column counts.
4. Use `.head()` to inspect the first few rows.

The table should contain 784 pixel columns plus one label column. The labels are categories, not pixels.


In [ ]:
# Build one table row per image and keep its category in a separate column.

# Supplied: read the binary files. The Pandas work begins below these two calls.
training_pixels, training_categories = read_mnist_idx(
    disk_files["train_images"],
    disk_files["train_labels"],
)
test_pixels, test_categories = read_mnist_idx(disk_files["test_images"], disk_files["test_labels"])

PIXEL_COLUMNS = [f"pixel_{i:03d}" for i in range(28 * 28)]

# Make a table from the pixels, then add one category label for each row.
official_training_frame = ## FINISH_ME ##
official_training_frame["label"] = training_categories
official_test_frame = ## FINISH_ME ##
official_test_frame["label"] = ## FINISH_ME ##

print("Training table (rows, columns):", official_training_frame.shape)
print("Test table (rows, columns):", official_test_frame.shape)

# Display the first five rows, showing the first four pixel columns and the label column.

## FINISH_ME ##

In [ ]:
# Inspect the dimensions and category counts of the loaded tables.

# Inspect the table as data: one row is one image; the final column is its label.
print("Training table shape:", official_training_frame.shape)
print("Test table shape:", official_test_frame.shape)
print("Number of pixel columns:", len(PIXEL_COLUMNS))

# Counting labels is a quick way to see which categories the sample contains.

# Hint: .value_counts().sort_index() allow you to see the counts of each label in order.

## FINISH_ME ##

**Explanation:**

`pd.DataFrame` creates one row per image and gives its 784 pixel columns names.
Assigning `frame["label"]` adds the label column at the end. `.shape` reports rows and columns;
`.head()` previews the first rows. The full training table has 60,000 rows and 785 columns.
Its pixel values are still integers from 0 to 255.


### 1.2 Split rows and build a PyTorch Dataset


**Part A: select the rows.** The shuffled row positions are supplied. Use `.iloc[positions]`
to select the training and validation rows, and keep the official test table separate.
`.iloc` uses row positions, so we do not need to reset or copy the DataFrame index.


In [ ]:
# Use one shared row permutation so images and their labels stay paired.

# Supplied: shuffled row positions, using the same fixed split for every model.
randomness = torch.Generator().manual_seed(SEED)
row_order = torch.randperm(len(official_training_frame), generator=randomness).tolist()

# Select the first 50,000 rows for training and the remaining 10,000 for validation.
train_indices =  ## FINISH_ME ##
validation_indices = ## FINISH_ME ##

# .iloc selects rows by POSITION. Keep the official test data separate.
train_frame = official_training_frame.iloc[train_indices]
validation_frame = official_training_frame. ## FINISH_ME ##
test_frame = official_test_frame

print("Training rows:", len(train_frame))
print("Validation rows:", len(validation_frame))
print("Test rows:", len(test_frame))


**Part B: convert the table into tensors**

Follow the physical meaning of each object:

1. Separate the 784 pixel columns from the label column.
2. Extract their numerical values from the DataFrame.
3. Store pixel intensities as floating-point numbers and divide them by 255.
4. Reshape the pixels into images: one batch axis, one greyscale channel and two spatial axes.
5. Store the labels separately as integers.

The row order must remain the same in both outputs. Inspect a two-image example before constructing
the full Dataset.


In [ ]:
# Convert tabulated intensities into floating-point image tensors.

def dataframe_to_tensors(frame):
    # 1. Select the input pixels, leaving out the category label.
    
    ## FINISH_ME ##

    # 2. Convert the pixels to floats, divide by 255, and reshape to [N, 1, 28, 28].

    ## FINISH_ME ##

    # 3. Keep the category labels as a separate integer tensor.
    label_values = frame["label"].to_numpy()
    labels = torch.tensor(label_values, dtype=torch.long)

    return images, labels

# Inspect a small example before putting the converter inside the Dataset.
example_images, example_labels = dataframe_to_tensors(train_frame.head(2))
print("Converted image shape:", example_images.shape)
print("Converted label shape:", example_labels.shape)


**Part C: make a small Dataset**

The conversion function now does the Pandas work. The class below only needs three methods:
`__init__` stores the tensors, `__len__` returns the number of images, and `__getitem__`
returns one image. Read these supplied methods, then create the validation and test Dataset objects.
Keep the supplied comma in `(image,)`; it makes a one-element tuple.


In [ ]:
# Give the DataLoader access to one image at a time.

class MNISTFrameDataset(Dataset):
    def __init__(self, frame):
        # Convert once when constructing the Dataset, rather than on every request.
        self.images, self.labels = dataframe_to_tensors(frame)

    def __len__(self):
        # One example means one image.
        return len(self.images)

    def __getitem__(self, index):
        # A one-element tuple matches the batch-unpacking convention in the training loop.
        return (self.images[index],)

train_dataset = MNISTFrameDataset(train_frame)
validation_dataset = MNISTFrameDataset( ## FINISH_ME ## )
test_dataset = MNISTFrameDataset( ## FINISH_ME ## )


In [ ]:
# Inspect the same images after the DataFrame-to-Dataset conversion.

# Keep the images and labels in the same row order for later comparisons.
train_images = train_dataset.images
train_labels = train_dataset.labels
validation_images = validation_dataset.images
validation_labels = validation_dataset.labels
test_images = test_dataset.images
test_labels = test_dataset.labels

# Read each dimension as: images, channels, rows, columns.
print("Training images:", ## FINISH_ME ##)
print("Validation images:", ## FINISH_ME ##)
print("Test images:", test_images.shape)
print("One Dataset image:", train_dataset[0][0].shape)
print("First image category:", CLASS_NAMES[train_labels[0].item()])


**Explanation:**

Row selection defines which images belong to each sample. Removing the label column leaves the
model inputs. Reshaping changes how the same 784 numbers are indexed; it does not change their values
or move pixels between images.

The Dataset stores the converted tensors. Its length counts images, and asking for one index returns
one image. Labels stay aligned with those images for later interpretation.


### 1.3 Build the training and validation DataLoaders


A Dataset provides one example at a time. A DataLoader collects examples into batches for the model.

Build the training and validation loaders separately. Shuffle the training sample to vary the order
of gradient updates. Keep validation in a fixed order because it is a measurement, not a parameter update.
Inspect the first batch: identify its image axis, channel axis and two spatial axes.


In [ ]:
# Group training examples into batches for successive parameter updates.

# Shuffle training images so each epoch presents them in a different order.
training_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, batch_size= ## FINISH_ME ## , shuffle=True, generator= ## FINISH_ME ##,
)


In [ ]:
# Use separate batches to measure the model without updating it.

# Keep validation images in order; they are used for measurement, not weight updates.
validation_generator = torch.Generator().manual_seed(SEED)
validation_loader = DataLoader(
    validation_dataset, batch_size=## FINISH_ME ##, shuffle=False, generator=## FINISH_ME ##,
)


In [ ]:
# Read the batch shape as a statement about the data entering the model.

first_batch = next(iter(train_loader))
batch_images = first_batch[0]
print("Batch shape:", ## FINISH_ME ##)
print("Pixel range:", batch_images.min().item(), "to", batch_images.max().item())


In [ ]:
# Inspect a batch, then restore the shared shuffle sequence.

# Interpret the inspected batch before beginning a fit.
print("Images in this batch:", ## FINISH_ME ##)
print("Image shape after the batch axis:", tuple(batch_images.shape[1:]))

# Inspection consumed a batch; reset the generator so experiments share a shuffle order.
train_loader.generator.manual_seed(SEED)


**Complete the image viewer — a link between the table and the picture**

The reshaping helper returns an image and its title. The drawing helper can either make its own figure
or draw on an axis supplied by another plot. Both expect intensities on the original 0–255 scale.

Use one DataFrame row to check that you recognise the digit. Later reconstruction plots reuse this
viewer, converting the model's 0–1 intensities back to the viewer's input scale.


In [ ]:
# Turn the same 784 intensity values back into a recognisable image.

def reshape_to_image(data, row_index=0):
    """Reshape one image and return (image, title)."""
    row = data.iloc[row_index] if isinstance(data, pd.DataFrame) else data
    title = "MNIST image"

    # Check if input was a Pandas Series
    if isinstance(row, pd.Series):
        title = CLASS_NAMES[int(row["label"])] if "label" in row else f"DataFrame row {row.name}"
        # 
        row = ## FINISH_ME ##

    # Check if input was a PyTorch Tensor
    if isinstance(row, torch.Tensor):
        # Hint: .detach() and .cpu() are needed to convert a tensor to a NumPy array.
        row = ## FINISH_ME ##

    image = np.asarray(row, dtype=np.float32).reshape(28, 28)

    image = image / 255.0
    return image, title


def show_mnist_row(data, row_index=0, ax=None, cmap="gray"):
    """Draw one image; pass ax to reuse this method inside a larger figure."""
    image, title = reshape_to_image(data, row_index=row_index)

    # True if the caller did not supply an axis; we will create our own figure and axis.
    own_figure = ## FINISH_ME ##

    if own_figure:
        _, ax = plt.subplots(figsize=(3, 3))

    # Always show the image
    ax.imshow(image, cmap=cmap, vmin=0, vmax=1, interpolation="nearest")

    if own_figure:
        ax.set(title=title, xlabel="Pixel column", ylabel="Pixel row")
        plt.show()
    return image


# Read the pixels directly from one DataFrame row.
dataframe_image = show_mnist_row(train_frame, row_index=0)
# A pixel-only Series also works: show_mnist_row(train_frame.iloc[0][PIXEL_COLUMNS])


**Explanation:**

`train_loader` batches the training Dataset and shuffles its image order. The validation loader
visits its Dataset in order. `next(iter(train_loader))` gets a batch; element 0 contains the images.
With the default `BATCH_SIZE`, the first batch has 256 images, each with one 28 × 28 channel. Section 2 uses this same training loader.


**Question:** Why must the same indices select images and labels, even though the plain AE does not use labels?


**Answer:**

`## FINISH_ME ##`

## 2.0 AE definition and training


### 2.1 Build and inspect the encoder


Think of the image as a 784-component measurement vector. The encoder maps it to a shorter
vector of learned coordinates:

**784 intensities → 512 features → 256 features → 128 features → latent coordinates**

Each dense layer applies an affine map, $h=Wx+b$. ReLU replaces negative components by zero,
allowing a sequence of layers to represent a nonlinear map. The final latent layer has no activation,
so its coordinates can be positive or negative.

Complete the encoder, then follow the printed shapes one layer at a time. The first axis always counts images; flattening should
combine the pixels within an image, never combine different images.


In [ ]:
# Follow the dimensions of the learned map from image pixels to latent coordinates.

def build_encoder(in_channels=1):
    return nn.Sequential(
        # Flatten only the image dimensions; retain the batch axis.
        nn.Flatten(),
        # Dense layers mix input components; ReLU makes the map nonlinear.
        nn.Linear(in_channels * 784,  ## FINISH_ME ##),
        nn.ReLU(),

        ## FINISH_ME ## Build a model which descends from 784, 512, 256, 128, to LATENT_DIM

        # The final values are the unconstrained latent coordinates.
        nn.Linear(128, LATENT_DIM),
    )

# Now we need to make sure the encoder is working

# First pass some images through the encoder and then print the shape of each layer's output

with torch.no_grad():
    features = train_images[:4]
    for layer in build_encoder():

        # The next features takes the output from the previous layer as input
        features =  ## FINISH_ME ##

        # Print Layer Name(type) and shape of the output tensor
        print(f"{type(layer).__name__:12s} -> {tuple(## FINISH_ME ##)}")


**Question:** Why does the final encoder layer have no activation function before the latent space?

**Answer:**

`## FINISH_ME ##`

Complete the decoder by reversing the encoder’s hidden-layer sizes: map the latent code through
128, 256 and 512 features, then produce 784 pixel values. Use ReLU after each hidden layer and
sigmoid after the final linear layer. Restore the image shape while keeping the batch axis.


In [ ]:
# Map latent coordinates back to one predicted intensity per pixel.

def build_decoder():
    # Expand the short code into enough values to describe an image.
    return nn.Sequential(
        nn.Linear(LATENT_DIM, 128),

        # Build a model which ascends from LATENT_DIM, 128, 256, 512, to 784

        # This is the opposite of the encoder

        # Map the predicted intensities into the input range, 0 to 1.
        ## Add a layer to map the predicted intensities into the input range, 0 to 1. Hint: use nn.Sigmoid()
        # Restore channel, row and column axes without changing the batch.
        nn.Unflatten(1, (1, 28, 28)),
    )


**Explanation:**

The last linear layer creates 784 pixels for each image. `Unflatten(1, (1, 28, 28))` reshapes the feature axis while preserving the batch axis. The output is `[batch, 1, 28, 28]`, matching the input.


**Supplied initialisation — He weights for the hidden layers**

He normal initialisation gives hidden weights a standard deviation of $\sqrt{\frac{2}{n_{\mathrm{in}}}}$.
The hidden layers use ReLU, matching the He gain. The final code projection,
sigmoid output projection and VAE posterior heads use unit gain. All Linear-layer biases start at zero.

This initialisation runs when a fresh model is constructed. DEC keeps the trained AE weights it copies.
Initialisation can help optimisation; the KL weighting and training also determine whether latent coordinates remain active.
See [PyTorch's initialisation functions](https://docs.pytorch.org/docs/2.8/nn.init.html#torch.nn.init.kaiming_normal_).


In [ ]:
# Set an appropriate initial variance for each layer's weights.

def initialise_he(layers):
    """Choose starting weight scales appropriate for the following activation."""
    linear_layers = [layer for layer in layers if isinstance(layer, nn.Linear)]

    # ReLU removes negative activations; He scaling compensates for that loss of variance.
    for layer in linear_layers[:-1]:
        nn.init.kaiming_normal_(layer.weight, mode="fan_in", nonlinearity="relu")
        nn.init.zeros_(layer.bias)

    # The last layer is a linear code or a sigmoid input, so it uses unit gain.
    output_layer = linear_layers[-1]
    nn.init.kaiming_normal_(output_layer.weight, mode="fan_in", nonlinearity="linear")
    nn.init.zeros_(output_layer.bias)


Complete `reconstruct`: first encode the images, then decode their codes. Calling `model(images)` runs `forward`, which delegates to this method.


In [ ]:
# Join the encoder and decoder into one model.

class DenseAE(nn.Module):
    def __init__(self, in_channels=1):
        super().__init__()
        # The encoder and decoder hold the trainable parameters of the two maps.
        self.encoder = ## FINISH_ME ##
        self.decoder = ## FINISH_ME ##

        # Initialise once at construction; never reinitialise a fitted model before evaluation.
        initialise_he(self.encoder)
        initialise_he(self.decoder)

    def encode(self, images):
        # Map every image to one row of latent coordinates.
        return self.encoder(images)

    def reconstruct(self, images):
        # Compress the measurements, then reconstruct intensities from that code.
        
        ## FINISH_ME ##

        # The autoencoder output is the decoder's reconstruction of the encoder's code.
        
        return ## FINISH_ME ##

    # PyTorch calls forward when the model is called on a batch.
    def forward(self, images):
        return self.reconstruct(images)


**Explanation:**

`self.encode(images)` returns one latent vector per image; `self.decoder` maps those vectors back to image tensors. No category label enters either operation.


### 2.2 Use the supplied reconstruction mathematics


The decoder predicts a greyscale intensity $\hat{x}_{ip}$ for each input intensity $x_{ip}$.
The supplied reconstruction objective is mean binary cross-entropy:

$$
L_{\mathrm{rec}}
= -\frac{1}{BP}\sum_{i=1}^{B}\sum_{p=1}^{P}
\left[
x_{ip}\log\hat{x}_{ip}
+ (1-x_{ip})\log(1-\hat{x}_{ip})
\right],
\qquad P=784.
$$

This expression leaves one scalar for a whole batch. A smaller value means closer agreement under
this objective. Here it is a loss for greyscale reconstruction, not a probability that an image belongs
to a digit category.

Compare this with a least-squares residual sum: both aggregate discrepancies, but they weight
different kinds of pixel disagreement differently.


In [ ]:
# Measure a simple reconstruction objective and inspect the forward calculation.

def ae_losses(rebuilt, images):
    """Return the batch's mean pixel reconstruction loss."""
    # Average over both images and pixels: the result is one differentiable scalar.
    reconstruction = F.binary_cross_entropy(rebuilt, images)

    # Every model uses the same reporting convention: total, reconstruction, extra term.
    return reconstruction, reconstruction, 0.0

# Follow eight images through an untrained model.
check_ae = DenseAE()
with torch.no_grad():
    check_images = train_images[:8]
    check_codes = check_ae.encode(check_images)
    check_rebuilt = check_ae(check_images)

    # A constant prediction provides a simple reference for interpreting the loss.
    grey_prediction = torch.full_like(check_images, 0.5)
    grey_loss = ae_losses(grey_prediction, check_images)[0]

print("Input images:", check_images.shape)
print("Encoded vectors:", check_codes.shape)
print("Reconstructed images:", check_rebuilt.shape)
print(f"All-grey BCE: {grey_loss:.4f}; compare with log(2) = {np.log(2):.4f}")


**Question:** Does a code with `LATENT_DIM` coordinates limit the AE to `LATENT_DIM` different images? Why is the all-grey BCE constant?


**Answer:**

`## FINISH_ME ##`

### 2.3 Learn one batch at a time


A batch supplies examples for one update. An epoch visits the whole training sample.

Follow one update in order: calculate a prediction, calculate its loss, differentiate the loss
with respect to the model parameters, then change those parameters. Gradients from the preceding
update must be cleared first.

Keep the loss as a tensor until differentiation is finished. A Python number is useful for plotting
but cannot carry the chain of derivatives. These models receive images only; digit labels remain
separate for interpretation.


Complete the two lines after the loss. Use `loss.backward()` and `optimiser.step()` in the correct order; explain why that order matters.


In [ ]:
# Make the four operations of a gradient update explicit.

def forward_one_batch(model, images, loss_function):
    """Evaluate a batch and retain the total loss tensor for differentiation."""
    # Call the model with the images; keep digit labels out of training.

    # Get the output from calling the model
    output = ## FINISH_ME ##(images)

    loss, reconstruction, extra = loss_function(output, images)

    # Keep loss as a tensor. Only the reported reconstruction value becomes a number.
    return loss, reconstruction.item(), extra

def train_one_batch(model, images, optimiser, loss_function):
    """Use one batch to make one parameter update."""
    # 1. Remove gradients left by the preceding batch.
    optimiser.zero_grad()

    # 2. Predict the images and calculate one scalar loss.
    loss, reconstruction, extra = forward_one_batch(
        model, images, loss_function
    )

    ## FINISH_ME ##

    # Apply the chain rule and update the model parameters using their gradients.

    # The plotting history needs a number, not the full differentiation graph.
    return loss.item()


**Explanation:**

Back-propagation fills parameter `.grad` fields. The optimiser then reads them to update weights. Updating first would use missing or stale gradients. `zero_grad()` prevents accumulation from the previous batch.


### 2.4 Train across batches and epochs


Use the separate training and validation loaders from Section 1.3.

For each epoch, the training loop updates parameters once per batch. The validation loop then
measures the fitted model without changing it. Evaluation mode and disabling gradient recording
serve different purposes; both are used during validation.

The history stores every batch loss and one mean per epoch. When combining batch means, weight
each by its number of images: a short final batch should contribute less than a full batch.
The reconstruction and any additional penalty are recorded separately so their effects remain visible.


In [ ]:
# Repeat batch updates and record measurements over complete epochs.

@torch.no_grad()
def validate_one_epoch(model, validation_loader, loss_function, history):
    # Validation: visit every batch without changing the weights.

    # Total loss, reconstruction and extra terms are accumulated over all batches.
    validation_total = 0.0
    reconstruction_total = 0.0
    extra_total = 0.0

    for (images,) in validation_loader:

        # Forward the batch and calculate the loss, but do not backpropagate.
        loss, reconstruction, extra = forward_one_batch(model, images, loss_function)

        # Retain individual batch measurements as well as the epoch mean.
        history["validation_batch"].append(loss.item())
        validation_total += loss.item() * len(images)
        reconstruction_total += reconstruction * len(images)
        extra_total += float(extra) * len(images)

    # Compute the mean loss over all validation images and record it for plotting.
    history["validation"].append(validation_total / len(validation_loader.dataset))
    history["reconstruction"].append(reconstruction_total / len(validation_loader.dataset))
    history["extra"].append(extra_total / len(validation_loader.dataset))


def train_all(model, train_loader, validation_loader, loss_function, epochs):
    """Visit every training and validation batch each epoch; record batch and epoch losses."""
    optimiser = torch.optim.AdamW(model.parameters(), lr=0.001)

    train_loader.generator.manual_seed(SEED)     # Repeat the shuffle order for each model.

    # Record the loss of every batch and the mean loss of every epoch.
    history = {
        "batch": [],
        "validation_batch": [],
        "train": [],
        "validation": [],
        "reconstruction": [],
        "extra": [],
    }

    for epoch in range(epochs):

        # Training: update once per batch and keep every batch's loss.
        model.train()
        train_total = 0.0
        for (images,) in train_loader:

            batch_loss = train_one_batch(model, images, optimiser, loss_function)
            history["batch"].append(batch_loss)
            # The short final batch contributes only its own number of images.
            train_total += batch_loss * len(images)

        history["train"].append(train_total / len(train_loader.dataset))
        model.eval()

        # Validation: visit every batch without changing the weights.
        validate_one_epoch(model, validation_loader, loss_function, history=history)

        print(f"Epoch {epoch + 1}/{epochs}: training = {history['train'][-1]:.4f}, "
              f"validation = {history['validation'][-1]:.4f}")
    return history


In [ ]:
# Observe the effect of one gradient step on a separate demonstration model.

# Use a separate model to observe one update without changing the experimental AE.
check_model = DenseAE()

weights_before = check_model.encoder[1].weight.detach().clone()

check_optimiser = torch.optim.Adam(check_model.parameters(), lr=0.001)
check_loss = train_one_batch(
    check_model, train_images[:8], check_optimiser, ae_losses
)

# A parameter change makes the effect of the gradient step visible.
weights_after = check_model.encoder[1].weight.detach()

mean_change = ## FINISH_ME ##
# Calculate the mean absolute change across the first layer's weights after this batch update.
# Hint weights_after and weights_before are both tensors; use .abs() and .mean() to get the mean absolute change.

print(f"Loss for this batch: {check_loss:.4f}")
print(f"Mean absolute change in the first layer: {mean_change.item():.6f}")


In [ ]:
# Fit the plain dense AE used later in the anomaly investigation.

torch.manual_seed(SEED)
ae = DenseAE()
ae_history = train_all(ae, train_loader, validation_loader, ae_losses, AE_EPOCHS)
print("Recorded training batches:", len(ae_history["batch"]))
print("Recorded validation batches:", len(ae_history["validation_batch"]))


### 2.5 Reporting helpers after the first fit


The first AE has now been fitted. These reporting functions help you inspect learning
curves and reconstructions. Complete the marked expressions before using the helpers.

Read each plot as an experimental diagnostic. A decreasing objective is useful evidence, but
does not on its own establish that the representation is useful for every later task.
Section 6 deliberately supplies no anomaly-analysis plotting functions.


In [ ]:
# Display inputs, reconstructions and squared pixel errors with consistent scales.

@torch.no_grad()
def show_examples(model, images, title):
    """Build an input/reconstruction/error grid using the Section 1.3 image viewer."""
    model.eval()

    rebuilt = model.reconstruct(images)

    pixel_errors = ## FINISH_ME ##
    # Hint: the pixel error is the squared difference between the _rebuilt_ and original _images_.


    fig, axes = plt.subplots(3, len(images), figsize=(12, 5), squeeze=False, layout="constrained")

    # Loop over the columns of the grid, showing the input, reconstruction and pixel error for each image.
    for column in range(len(images)):
        # The viewer expects 0–255 pixels; the model uses 0–1 tensors.
        show_mnist_row(images[column, 0] * 255, ax=axes[0, column])
        show_mnist_row(rebuilt[column, 0] * 255, ax=axes[1, column])
        show_mnist_row(pixel_errors[column, 0] * 255, ax=axes[2, column], cmap="inferno")

    # Label the rows and remove the ticks from every axis.
    axes[0, 0].set_ylabel("Input")
    axes[1, 0].set_ylabel("Rebuilt")
    axes[2, 0].set_ylabel("Squared error")

    for axis in axes.flat:
        axis.set(xticks=[], yticks=[])

    # Add a colour bar using the first error image; all error panels share the same scale.
    heatmap = axes[2, 0].images[-1]
    fig.colorbar(heatmap, ax=list(axes[2]), label="Squared pixel error", shrink=0.7)
    fig.suptitle(title)
    plt.show()



In [ ]:
# Read a few numerical measurements alongside the learning curves.

def summarise_training(history, name):
    """Print a few numbers that help interpret the learning curves."""
    training = history["train"]
    validation = history["validation"]

    # Compare the final measurement with a recent one, rather than one noisy batch.
    earlier_index = max(len(validation) - 6, 0)
    earlier_loss = validation[earlier_index]
    recent_change = validation[-1] - earlier_loss
    best_epoch = int(np.argmin(validation)) + 1

    print(name)
    print(f"  First training batch: {history['batch'][0]:.4f}")
    print(f"  Final training mean: {training[-1]:.4f}")
    print(f"  Validation: {validation[0]:.4f} initially, {validation[-1]:.4f} finally")
    print(f"  Lowest validation value: epoch {best_epoch}")
    print(f"  Recent validation change: {recent_change:+.4f}")

    # The plain AE has no extra penalty; the VAE and DEC have two objective terms.
    if any(history["extra"]):
        print(f"  Final reconstruction term: {history['reconstruction'][-1]:.4f}")
        print(f"  Final additional term: {history['extra'][-1]:.4f}")


### 2.6 Interpret your first model


Read the three panels together:

1. Batch losses show the fluctuations between successive gradient updates.
2. Epoch means compare fitting and validation.
3. Validation components show the reconstruction term and any additional penalty separately.

Training measurements are made while parameters change; validation uses the parameters at the end
of an epoch. Look for trends over several epochs rather than interpreting an isolated fluctuation.
The notebook retains the final parameters.


In [ ]:
# Use three separate helpers to build three panels in one learning-curve figure.

def plot_batch_losses(history, ax):
    """Plot the loss from every training batch on the first subplot."""

    ## FINISH_ME ##
    # Hint: use ax.plot() to plot the training loss from every batch, then set the title, x-label, y-label and legend.


def plot_epoch_losses(history, ax, compare=None):
    """Compare training and validation means on the second subplot."""
    epochs = range(1, len(history["train"]) + 1)
    ax.plot(epochs, history["train"], label="Training")
    ax.plot(epochs, history["validation"], label="Validation")
    if compare is not None:
        other_history, name = compare
        other_epochs = range(1, len(other_history["validation"]) + 1)
        ax.plot(other_epochs, other_history["validation"], "--", label=f"{name} validation")
    ax.set_title("Training and validation per epoch")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()


def plot_validation_components(history, ax):
    """Show the validation loss components on the third subplot."""
    epochs = range(1, len(history["validation"]) + 1)
    ax.plot(epochs, history["reconstruction"], label="Reconstruction")
    if any(history["extra"]):
        ax.plot(epochs, history["extra"], label="Extra penalty")
    ax.set_title("Validation loss components")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()


def plot_training(history, title, compare=None):
    """Draw three subplots in one figure, using one helper for each subplot."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
    plot_batch_losses(history, axes[0])
    plot_epoch_losses(history, axes[1], compare)
    plot_validation_components(history, axes[2])
    fig.suptitle(title)
    plt.show()


In [ ]:
# Compare the learning curves with actual reconstructed validation images.

# Three subplots: every training batch, training vs validation after each epoch, and the objective's
# parts.
plot_training(ae_history, "Dense AE")
summarise_training(ae_history, "Dense AE")
show_examples(ae, validation_images[:6], "Plain AE: validation reconstructions")


**Question:** What do the training and validation curves, recent validation change and reconstructed stroke shapes tell you? Quote your run’s values.


**Answer:**

`## FINISH_ME ##`


## 3.0 Implement a VAE using supplied mathematics


The ordinary AE assigns one latent vector to an image. A variational autoencoder assigns
a Gaussian distribution:

$$
q_{\phi}(z\mid x)
= \mathcal{N}\!\left(
\mu_{\phi}(x),
\operatorname{diag}\!\left(\sigma_{\phi}^{2}(x)\right)
\right).
$$

Two learned outputs give the mean $\mu$ and the log variance $\ell=\log\sigma^2$.
Both have one component per latent coordinate. Therefore $\sigma=\exp(\ell/2)$.

The supplied sampling expression is

$$
z=\mu+\exp(\ell/2)\epsilon,
\qquad \epsilon\sim\mathcal{N}(0,I).
$$

The random draw is in $\epsilon$. The mean and scale remain part of the differentiable calculation,
so the reconstruction loss can update the encoder.

The second supplied expression compares this Gaussian with the standard-normal prior:

$$
D_{\mathrm{KL}}(q_{\phi}\,\|\,p)
= \frac{1}{2B}\sum_{i=1}^{B}\sum_{j=1}^{d}
\left[
\mu_{ij}^{2}+\exp(\ell_{ij})-1-\ell_{ij}
\right].
$$

The workshop uses

$$
L=L_{\mathrm{rec}}+\frac{3}{784}D_{\mathrm{KL}}(q_{\phi}\,\|\,p).
$$

The KL expression already averages over images and sums over latent coordinates. Division by 784
makes its scaling comparable with a mean pixel reconstruction loss; the factor 3 sets its relative
importance. You are not expected to derive these Gaussian expressions. Your task is to construct
the model and explain the competing effects of the two terms.


In [ ]:
# Translate the supplied Gaussian sampling and KL expressions into tensor operations.

def reparameterise(mean, logvar):
    """Draw one latent vector for each image, retaining derivatives of mean and scale."""
    # logvar represents log(sigma squared), so half of it represents log(sigma).
    standard_deviation = torch.exp(0.5 * logvar)

    # Independent unit-normal noise supplies the randomness for every coordinate.
    noise = torch.randn_like(mean)
    return mean + standard_deviation * noise

def gaussian_kl(mean, logvar):
    """Measure departure from a standard-normal prior, averaged over images."""
    # Each term describes one coordinate of one image's Gaussian.
    coordinate_terms = mean.square() + logvar.exp() - 1 - logvar

    # First add coordinate contributions within each image.
    image_terms = coordinate_terms.sum(dim=1)

    # Then average images; this is still a tensor that can be differentiated.
    return 0.5 * image_terms.mean()

# Compare the supplied expression with two simple Gaussian examples.
zero = torch.zeros(4, LATENT_DIM)
print("KL for a Gaussian matching the prior:", gaussian_kl(zero, zero).item())
print("KL for unit means and unit variances:", gaussian_kl(torch.ones_like(zero), zero).item())


### 3.1 Complete the model, not just a formula


Reuse the dense encoder and decoder, then add separate outputs for the Gaussian mean and log variance.

| Method | Meaning |
|---|---|
| Posterior | Return the mean and log variance for each image. |
| Encode | Return the mean vector, giving a repeatable coordinate for a plot. |
| Sample codes | Draw one latent vector from each predicted Gaussian. |
| Forward | Decode a draw and return the reconstruction plus the Gaussian parameters. |

The inherited reconstruction method decodes the mean. Sampling and reconstructing from the mean
answer different questions. Evaluation mode does not remove a random draw explicitly requested by
the model.


Complete both posterior heads and the return expression in `forward`. Use the supplied methods
to trace the four paths in the table. The decoder receives sampled codes during training and mean
codes during deterministic reconstruction.


In [ ]:
# Predict a Gaussian mean and log variance for each input image.

class DenseVAE(DenseAE):
    def __init__(self):
        super().__init__()


        # Two learned heads describe different parameters of the same Gaussian.
        self.mean_head = ## FINISH_ME ##
        self.logvar_head = ## FINISH_ME ##

        # Each head maps LATENT_DIM features to LATENT_DIM Gaussian parameters.
        # The supplied reparameterise function draws a sample from those parameters.

        # Posterior heads are linear: use unit gain and zero biases.
        for head in (self.mean_head, self.logvar_head):
            nn.init.kaiming_normal_(head.weight, mode="fan_in", nonlinearity="linear")
            nn.init.zeros_(head.bias)

    def posterior(self, images):
        # First calculate the shared features for this batch of images.
        features = self.encoder(images)
        # Return one mean and one log variance for every latent coordinate.
        return self.mean_head(features), self.logvar_head(features)

    def encode(self, images):
        mean, logvar = self.posterior(images)
        # The mean gives a repeatable coordinate for representation plots.
        return mean

    def sample_codes(self, images):
        mean, logvar = self.posterior(images)
        return reparameterise(mean, logvar)

    # PyTorch calls forward when the model is called on a batch.
    def forward(self, images):
        mean, logvar = self.posterior(images)
        # A random draw is used when optimising the variational objective.
        codes = reparameterise(mean, logvar)

        # Return three model outputs: the decoded image, the mean and the log variance.
        # The decoder itself returns only the image; the loss also needs the Gaussian parameters.

        return ## FINISH_ME ##


**Explanation:**

Both heads read the same encoded features but learn different parameters. `forward` computes the posterior once so the returned parameters describe the distribution used for that reconstruction. `encode` returns only the mean, making inherited reconstruction deterministic. Calling `sample_codes` uses the same distribution but adds noise. None of these methods detaches the posterior from the gradient graph.


Combine mean pixel BCE with the supplied Gaussian KL, using the factor $3/784$ shown above.
Return the total, reconstruction term and weighted KL separately. Inspect their values and the tensor
shapes, then fit a fresh VAE with the existing training and validation loaders.


In [ ]:
# Keep reconstruction and the weighted Gaussian penalty visible as separate terms.

def vae_losses(output, images):
    """Combine reconstruction fidelity with a penalty for departing from the prior."""
    rebuilt, mean, logvar = output

    # This is the same mean pixel reconstruction objective as the ordinary AE.
    reconstruction = F.binary_cross_entropy(rebuilt, images)

    # KL already averages over images; divide by pixels and apply the chosen weight.
    number_of_pixels = images[0].numel()
    extra = 3.0 * gaussian_kl(mean, logvar) / number_of_pixels
    return reconstruction + extra, reconstruction, extra


**Explanation:**

The Gaussian helper produces a mean KL per image, already summed over latent coordinates.
Its weight in this experiment is $3/784$. Recording this weighted term separately lets you distinguish
better reconstruction from a closer match to the prior.


In [ ]:
# Inspect the VAE's outputs and compare two random latent draws.

# Inspect the three outputs produced by one VAE forward calculation.
check_vae = DenseVAE()
check_images = train_images[:8]
rebuilt, mean, logvar = check_vae(check_images)
total, reconstruction, extra = vae_losses((rebuilt, mean, logvar), check_images)

print("Reconstruction:", rebuilt.shape)
print("Gaussian means:", mean.shape)
print("Gaussian log variances:", logvar.shape)
print(f"Total loss: {total.item():.4f}")
print(f"Reconstruction term: {reconstruction.item():.4f}")
print(f"Weighted KL term: {extra.item():.4f}")

# Two samples illustrate stochastic encoding; reconstruction from the mean is repeatable.
with torch.no_grad():
    first_sample = check_vae.sample_codes(check_images)
    second_sample = check_vae.sample_codes(check_images)
    sample_difference = (first_sample - second_sample).abs().mean()

print(f"Mean difference between two latent draws: {sample_difference.item():.4f}")


In [ ]:
# Fit a fresh VAE using the existing training and validation batches.

torch.manual_seed(SEED)
vae = DenseVAE()
vae_history = train_all(vae, train_loader, validation_loader, vae_losses, VAE_EPOCHS)


In [ ]:
# Read the two objective terms alongside reconstructions from mean latent codes.

# The third subplot shows two loss components: reconstruction and the weighted KL penalty.
plot_training(vae_history, "Dense VAE: reconstruction + 3 KL / 784")
summarise_training(vae_history, "Dense VAE")
show_examples(vae, validation_images[:6], "VAE: reconstruction from mean codes")


**Question:** Why can a lower total VAE objective coexist with worse reconstructed detail? Why compare components?


**Answer:**

`## FINISH_ME ##`

**Question:** If the objective is written as $L=L_{\mathrm{rec}}+\beta D_{\mathrm{KL}}$,
what coefficient $\beta$ multiplies the mean-per-image KL in our code? Explain the role of
the division by the number of pixels.


**Answer:**

`## FINISH_ME ##`

## 4.0 Compare all digit classes in latent space


Make one figure for the AE and one for the VAE. Each has training on the left and validation on the
right. Show every digit class with the same colour convention in both panels.

Use two existing latent coordinates. Every encoded image still has all of its latent coordinates;
the plot displays only two of them. This distinction also matters when the latent dimension is three.

These are learned coordinates rather than fixed physical observables. The two models can choose
different scales and orientations, so compare their structure rather than matching numerical
positions between models.


### 4.1 Encode all training and validation images



Pass the complete training and validation tensors through each fitted encoder. Keep two sets of vectors
for the AE and two for the VAE. `eval()` and `torch.no_grad()` let us examine the models without training them further.


In [ ]:
# Collect a deterministic representation of every image for comparison.

ae.eval()
vae.eval()
with torch.no_grad():

    # Call encode() on each fitted model to obtain one deterministic latent vector per image.

    ae_train_vectors = ## FINISH_ME ##
    ae_validation_vectors = ## FINISH_ME ##
    vae_train_vectors = ## FINISH_ME ##
    vae_validation_vectors = ## FINISH_ME ##

print("AE vectors (training, validation):", ae_train_vectors.shape, ae_validation_vectors.shape)
print("VAE vectors (training, validation):", vae_train_vectors.shape, vae_validation_vectors.shape)


### 4.2 Put the vectors and labels into DataFrames


Make a DataFrame for each set of vectors. Columns `0`, `1`, and so on are latent coordinates;
`"label"` identifies the digit class. Use `.to_numpy()` when attaching labels so their row order
matches the vectors, even though the original image DataFrames retain shuffled index labels.


In [ ]:
# Attach category labels after encoding; labels do not enter these encoders.

# Each row contains one image's latent coordinates, followed by its digit label.
ae_training_latent = pd.DataFrame(ae_train_vectors.numpy())
ae_training_latent["label"] = train_frame["label"].to_numpy()
ae_validation_latent = pd.DataFrame(ae_validation_vectors.numpy())
ae_validation_latent["label"] = validation_frame["label"].to_numpy()

vae_training_latent = ## FINISH_ME ##
vae_training_latent["label"] = ## FINISH_ME ##
vae_validation_latent = ## FINISH_ME ##
vae_validation_latent["label"] = ## FINISH_ME ##


Each DataFrame row corresponds to one image, with its latent vector and category label.
The AE provides its learned code; the VAE's `encode()` provides its mean code.
Both training DataFrames contain all 50,000 training images, and both validation DataFrames contain all 10,000 validation images.


### 4.3 Plot all categories with one shared method



Inside the plotting method, loop over all ten categories. Select the rows for the current label in each
DataFrame, then draw those rows on the corresponding subplot. Repeat until every category is included.
Training and validation share axis limits within each figure so their patterns are easy to compare.


In [ ]:
# Display all categories with one shared plotting method.

def plot_latent(training_latent, validation_latent, title, dimensions=(0, 1)):
    """Show all categories: training on the left, validation on the right."""
    x_dimension, y_dimension = dimensions
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)

    # Loop over every category and draw its training and validation points in the same colour.
    for class_label, class_name in enumerate(CLASS_NAMES):

        training_class = training_latent[training_latent["label"] == class_label]
        validation_class = validation_latent[validation_latent["label"] == class_label]
        colour = f"C{class_label}"           # Use the same category colour in every subplot.

        # Draw the training points with scatter.
        axes[0].scatter(## FINISH_ME ##, ## FINISH_ME ##,
                        color=colour, s=3, alpha=0.4, label=class_name)

        # Draw the validation points with scatter
        axes[1].scatter(validation_class[x_dimension], validation_class[y_dimension],
                        color=colour, s=3, alpha=0.4, label=class_name)

    axes[0].set_title("Training")
    axes[1].set_title("Validation")

    axes[0].set_xlabel(f"Latent coordinate {x_dimension}")
    axes[1].set_xlabel(f"Latent coordinate {x_dimension}")
    axes[0].set_ylabel(f"Latent coordinate {y_dimension}")
    axes[1].legend(title="Digit", loc="upper left", bbox_to_anchor=(1.02, 1))

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


Call the same method for each model. The first call produces the AE figure; the second produces the VAE figure.
Each figure shows training and validation side by side, with every category included.


In [ ]:
# Compare training and validation within each model's learned coordinate system.

plot_latent(ae_training_latent, ae_validation_latent, "AE latent space")
plot_latent(vae_training_latent, vae_validation_latent, "VAE latent space")


**Questions:** Which digit classes occupy distinct regions, and which overlap? Do the category
patterns look similar in training and validation? How do the AE and VAE representations differ?

Compare training and validation within each model. The AE and VAE learn their own coordinate scales
and orientations, so matching positions across the two figures need not represent the same features.


**Answer:**

`## FINISH_ME ##`


## 5.0 Integrate a new objective: clustering with reconstruction


Deep embedded clustering (DEC) refines codes towards learned centres. Our **IDEC-style** variant keeps
reconstruction to discourage loss of image detail: **mean pixel BCE + 0.1 × clustering KL**.
Standard DEC does not retain this reconstruction term. The AE remains fully dense.

Copy the trained AE and initialise ten centres using a Gaussian mixture on training codes. Standardise coordinates
using fixed training means and standard deviations, while continuing to feed raw codes to the decoder.

With a standardised code $z_i$ and centre $c_j$, define

$$
s_{ij}=\frac{1}{1+\lVert z_i-c_j\rVert^2},\qquad q_{ij}=\frac{s_{ij}}{\sum_k s_{ik}}.
$$

$q$ contains soft assignments: each image has a row of ten probabilities summing to one.
DEC sharpens the partition it starts with, so a large mixed cluster can keep attracting several digits.
Here we use **balanced targets**, a teaching variant suited to MNIST's roughly similar category frequencies.
This is an assumption about cluster sizes; it does not tell the model which digit belongs in which cluster.

Start with $w_{ij}=q_{ij}^2$, then alternate these two rescalings **20 times**:

$$
u_{ij}=\frac{w_{ij}}{\sum_n w_{nj}},\qquad
w_{ij}\leftarrow\frac{u_{ij}}{\sum_k u_{ik}}.
$$

Return $p=w$. Squaring emphasises confident assignments. Column rescaling counters oversized clusters;
row rescaling restores one probability distribution per image. Repeating both makes cluster masses
approach $N/K$, while keeping every row sum at one. Use all training images to build these targets,
then hold them fixed for one epoch. Minimise $D_{\mathrm{KL}}(p\|q)$ alongside reconstruction.
This stronger size prior would need reconsideration for data with very unequal category frequencies.
See [Sinkhorn matrix scaling](https://arxiv.org/abs/1306.0895) for the rescaling idea.

Classic DEC uses K-means initialisation. Here, a full-covariance Gaussian mixture can model elongated
groups in the pretrained code space. Select a locally consistent, approximately balanced fit from repeated starts. Its means seed DEC,
and its probabilities supply a short initial target. The Student's t assignments and reconstruction objective are unchanged.

Digit labels remain outside pretraining, mixture fitting, target generation, updates and stopping decisions.
See the supplied [GaussianMixture initialiser](https://scikit-learn.org/1.6/modules/generated/sklearn.mixture.GaussianMixture.html).

After balanced refinement, the supplied neighbour extension uses nearby training codes to build a final fixed target.
Both stages train the same fully dense model with the same reconstruction-plus-KL loss.


Complete `DenseDEC.forward` so it returns the reconstruction and the cluster probabilities.
Complete the reconstruction term in `dec_losses` using mean pixel BCE.

The balanced-target helper and clustering KL are supplied. Trace how the helper squares the
probabilities and alternates column and row normalisation 20 times. The `keepdim=True` arguments
keep the sums shaped so they can be applied to the full matrix.


**Supplied preparation for Section 5.** Collect the plain AE's complete training and validation
code sets in batches. Later, use the same collector for the fitted clustering model.


In [ ]:
# Collect complete code sets in batches while preserving image order.

@torch.no_grad()
def collect_codes(model, images):
    """Encode a complete image set in manageable batches, keeping its row order."""
    model.eval()
    batches = []

    # Batching limits working memory; concatenation restores the original image order.
    for batch in images.split(BATCH_SIZE):
        codes = model.encode(batch)
        batches.append(codes)

    return torch.cat(batches)

def collect_code_sets(model):
    """Keep training and validation representations as two separate measurements."""
    training_codes = collect_codes(model, train_images)
    validation_codes = collect_codes(model, validation_images)
    return {"Training": training_codes, "Validation": validation_codes}

ae_codes = collect_code_sets(ae)


In [ ]:
# Add an unsupervised grouping objective while retaining reconstruction.

class DenseDEC(nn.Module):
    """A copy of the fitted AE with learnable centres in its latent space."""

    def __init__(self, trained_ae, centres, code_mean, code_std):
        super().__init__()


        self.dec_ae = copy.deepcopy(trained_ae)     # a copy, so DEC training leaves the original AE untouched
        # Copy the centres so fitting does not overwrite the initialisation.
        self.KL_centres = nn.Parameter(centres.detach().clone())


        # Store the training reference scale without making it a parameter.
        self.register_buffer("code_mean", code_mean)
        self.register_buffer("code_std", code_std)

    def encode(self, images):
        """Standardised codes: each coordinate rescaled by the AE's training mean and standard deviation."""
        return (self.dec_ae.encode(images) - self.code_mean) / self.code_std

    def reconstruct(self, images):
        return self.dec_ae.reconstruct(images)

    def forward(self, images):
        # The decoder still uses the original AE coordinates.
        raw_codes = self.dec_ae.encode(images)

        # Express each coordinate relative to its training spread.
        codes = (raw_codes - self.code_mean) / self.code_std
        # Each column gives distance to one centre, using all latent coordinates.
        distances = torch.cdist(codes, self.KL_centres)
        squared_distances = distances.square()
        # Student's t kernel: similarity 1 / (1 + d²) is 1 at a centre and falls off slowly,
        # with heavy tails.
        similarities = 1 / (1 + squared_distances)
        # Row normalisation gives one cluster-probability distribution per image.
        probabilities = similarities / similarities.sum(dim=1, keepdim=True)

        # The forward output is the same as the AE's, plus the cluster probabilities for each image for the KL loss.

        return ## FINISH_ME ##


def target_distribution(probabilities):
    """Sharpen q, then balance cluster masses across the full set of images."""
    targets = probabilities.square()

    for _ in range(20):
        targets = targets / targets.sum(dim=0, keepdim=True)  # Rescale cluster columns.
        targets = targets / targets.sum(dim=1, keepdim=True)  # Restore image row sums to one.

    return targets

def dec_losses(output, images, targets):
    """Return (reconstruction + 0.1 × KL(p || q), reconstruction, weighted clustering KL)."""

    rebuilt, probabilities = output
    # Reconstruction keeps the code connected to the measured image.
    reconstruction = ## FINISH_ME ##

    # Hint: use the functional F.binary_cross_entropy to calculate the BCE between rebuilt & images

    # F.kl_div takes log-probabilities first and computes KL(targets || probabilities), averaged
    # per image.
    extra = 0.1 * F.kl_div(probabilities.log(), targets, reduction="batchmean")

    return reconstruction + extra, reconstruction, extra


**Explanation:**

Each row represents one image and each column a centre. Distances use fixed-standardised codes, but the decoder
receives raw codes. Row normalisation turns similarities into assignments. Squaring these assignments makes
confident predictions more influential. Repeated column and row rescaling discourages one mixed cluster from
remaining much larger than the others, while preserving a probability distribution for each image.
The balanced-size assumption comes from the exercise design, never from digit labels.
`F.kl_div(log(q), p)` computes the requested KL direction; its batchmean reduction averages over images.


**Supplied initialisation — choose the partition before sharpening it**

A high mixture likelihood can favour splitting a compact handwriting style and merging broader groups.
Sharpening that partition can reinforce the wrong grouping. For this roughly balanced MNIST exercise,
try 20 mixture starts and compare them using training data only.

For component sizes $n_0,\ldots,n_{K-1}$, measure relative size spread as

$$
b=\frac{\operatorname{sd}(n_0,\ldots,n_{K-1})}{N/K}.
$$

Keep fits within 5% of the smallest $b$. Among these similarly balanced fits, prefer the one with the
largest fraction of neighbouring AE codes assigned to the same component. This stops tiny size-score
differences from deciding between fits with different local consistency. Neither score uses digit labels.

The supplied helper performs this selection. Its balance prior is specific to this exercise:
unequal natural class frequencies would require a different selection rule.

For a short initial training stage, use balanced targets from the selected mixture's full probabilities.
This retains information from its covariance shapes while the dense encoder adapts to the Student's t kernel.
Then refresh targets from DEC itself, followed by the existing fixed-neighbour-target stage.
All stages use the same reconstruction-plus-KL loss and stay within `DEC_EPOCHS`.


In [ ]:
# Select a starting partition using only training geometry and cluster sizes.

# Fitting the initial mixture is expensive. Reuse this fit when rerunning DEC
# with the same pretrained AE and data. Refit it if either changes. This helper is supplied.

from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors

def find_training_neighbours(training_codes):
    """Find 15 neighbours in the original AE training space, excluding each row itself."""

    search = NearestNeighbors(n_neighbors=16).fit(training_codes.numpy())
    rows = search.kneighbors(training_codes.numpy(), return_distance=False)[:, 1:]
    return torch.tensor(rows, dtype=torch.long)


def fit_initial_mixture(training_codes):
    """Choose a balanced, locally consistent starting partition without digit labels."""
    values = training_codes.numpy()
    neighbour_rows = find_training_neighbours(training_codes)
    fits = []

    for start in range(20):
        mixture = GaussianMixture(
            n_components=NUM_CLASSES,
            covariance_type="full",
            n_init=1,
            max_iter=150,
            reg_covar=1e-4,
            random_state=SEED + start,
        ).fit(values)


        # These are mixture component IDs, obtained without known digit categories.
        assignments = mixture.predict(values)
        # Count images per component to quantify the size-balance assumption.
        sizes = np.bincount(assignments, minlength=NUM_CLASSES)
        if not mixture.converged_ or sizes.min() == 0:
            continue
        balance = sizes.std() / sizes.mean()
        # Nearby representations should usually receive the same component ID.
        agreement = (assignments[:, None] == assignments[neighbour_rows.numpy()]).mean()
        fits.append({"model": mixture, "start": start, "balance": balance, "agreement": agreement})

    if not fits:
        raise RuntimeError("No converged mixture with ten populated components was found.")

    # Among similarly balanced fits, prefer stronger agreement between neighbours.
    smallest_spread = min(fit["balance"] for fit in fits)
    balanced_fits = [fit for fit in fits if fit["balance"] <= 1.05 * smallest_spread]
    chosen = max(balanced_fits, key=lambda fit: fit["agreement"])
    print("Selected mixture start:", chosen["start"])
    print(f"Relative cluster-size spread: {chosen['balance']:.3f}; "
          f"neighbour agreement: {100 * chosen['agreement']:.1f}%")
    return chosen["model"], neighbour_rows


In [ ]:
# Initialise clustering from the fitted AE, keeping the original AE unchanged.

# Fit only the AE's training codes; digit labels are not inputs to this fit.
code_mean = ae_codes["Training"].mean(dim=0)
code_std = ae_codes["Training"].std(dim=0).clamp_min(1e-6)

scaled_codes = (ae_codes["Training"] - code_mean) / code_std

# Select using training cluster sizes and neighbour agreement, never digit labels.
cluster_initialiser, initial_neighbour_rows = fit_initial_mixture(scaled_codes)
mixture_probabilities = torch.tensor(cluster_initialiser.predict_proba(scaled_codes.numpy()), dtype=torch.float32)
mixture_targets = target_distribution(mixture_probabilities)
centres = torch.tensor(cluster_initialiser.means_, dtype=torch.float32)
dec = DenseDEC(ae, centres, code_mean, code_std)
print("Initial mixture converged:", cluster_initialiser.converged_)


# Row numbers keep each shuffled image paired with its own DEC target.
row_numbers = torch.arange(len(train_images))
dec_loader = DataLoader(TensorDataset(
    train_images,
    row_numbers,
), batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED))


### 5.1 Implement refinement against fixed targets


The clustering loss changes the organisation of the latent vectors while reconstruction keeps them
connected to the input images. Follow the three target stages in the training loop:
mixture targets, refreshed DEC targets, then fixed neighbour targets.

At every stage, a batch's row numbers identify which full-sample targets belong to its images.
The diagnostic functions count occupied clusters and measure the spread of the coordinates.
Those quantities describe the representation; they do not use digit labels.

An empty cluster or a severe reduction in coordinate spread can indicate a failed fit. Inspect
these diagnostics when interpreting the result: this loop records them but does not automatically
reject an epoch or restore earlier weights. A compact cloud alone is not evidence of failure.
Digit labels are used only afterwards to investigate which categories were grouped together.


In [ ]:
# Measure cluster membership, coordinate spread and objective components.

@torch.no_grad()
def cluster_probabilities(model, images):
    """Soft cluster assignments q for many images, batch by batch: [N, 10]."""
    batches = []
    for batch in images.split(BATCH_SIZE):
        rebuilt, probabilities = model(batch)
        batches.append(probabilities)
    return torch.cat(batches)

@torch.no_grad()
def dec_training_status(model, images):
    """Measure assignments and code spread without using digit labels."""
    model.eval()
    probabilities = cluster_probabilities(model, images)
    # Choose the most probable component; this integer is not a digit label.
    assignments = probabilities.argmax(dim=1)
    counts = torch.bincount(assignments, minlength=model.KL_centres.shape[0])
    # Coordinate spread helps distinguish a compact representation from collapse.
    spread = collect_codes(model, images).std(dim=0)
    return probabilities, assignments, counts, spread

@torch.no_grad()
def measure_dec(model, images):
    """The DEC objective on a set of images, with targets sharpened from those same images' assignments."""
    reconstruction_sum = 0
    for batch in images.split(BATCH_SIZE):
        rebuilt = model.reconstruct(batch)
        # Combine batch means with the correct weight for the last, shorter batch.
        reconstruction_sum += float(F.binary_cross_entropy(rebuilt, batch)) * len(batch)
    probabilities = cluster_probabilities(model, images)
    # Refresh self-generated targets only between complete epochs.
    targets = target_distribution(probabilities)     # self-generated: a consistency check, not a fixed answer
    reconstruction = reconstruction_sum / len(images)
    extra = 0.1 * F.kl_div(probabilities.log(), targets, reduction="batchmean").item()
    return reconstruction + extra, reconstruction, extra

def record_dec(model, history):
    """Fixed-weight training-sample and validation measurements for the DEC plots."""
    model.eval()

    # The DEC objective is measured on a subset of the training images to save time.
    train = measure_dec(model, train_images[:10_000])
    validation, recon, extra = measure_dec(model, validation_images)

    history["train"].append(train[0])
    history["validation"].append(validation)
    history["reconstruction"].append(recon)
    history["extra"].append(extra)


Return a list of batch losses. Each batch from `dec_loader` is `(images, rows)`; train using `dec_losses(output, images, targets[rows])`. Use the update pattern learned in Section 2.


In [ ]:
# Update the clustering model against one fixed target per training image.

def train_dec_epoch(model, optimiser, targets):
    """One DEC epoch: the same four-step update as train_one_batch, towards this epoch's fixed targets."""
    model.train()
    batch_losses = []
    for images, rows in dec_loader:

        optimiser.zero_grad()
        output = model(images)
        # The row index survives shuffling, so each image keeps its own target.
        
        batch_targets = targets[rows]
        
        loss, reconstruction, extra = dec_losses(output, images, batch_targets)

        # Apply the chain rule and update the model parameters using their gradients.

        ## FINISH_ME ##

        batch_losses.append(loss.item())

    return batch_losses


**Explanation:**

The row indices refer to the full training dataset, so shuffling does not break target pairing.
Clear gradients, evaluate the reconstruction and clustering losses, back-propagate, then update the AE
and cluster centres. Targets were calculated without gradients and remain fixed throughout the epoch.
The centres own their storage, so learning does not overwrite the saved mixture initialisation.


**Supplied extension — preserve local digit structure**

A single centre can mix handwriting styles even when nearby AE codes describe the same kind of image.
After balanced DEC settles, use neighbours in the **original AE training codes** to correct some of these mistakes.

The supplied helper finds 15 neighbours per training image. Within each current cluster, the quarter with the
largest ratio of first-choice to second-choice probability become anchors. Their one-hot values encode
**the model's own cluster IDs**, with no digit labels.

Start from $P^{(0)}=q$. Repeat 40 times:

$$
P_i^{(t+1)}=\frac{1}{15}\sum_{n\in\mathcal N(i)}P_n^{(t)},\qquad
P_i^{(t+1)}=e_{\hat c_i}\quad\text{for anchored rows }i.
$$

Then hold these targets fixed for a short final training phase. Images, neighbours and model assignments
are the only inputs. The dense encoder, decoder and reconstruction-plus-KL loss stay the same.
This is an extension of our teaching variant, rather than standard DEC.
The supplied [nearest-neighbour search](https://scikit-learn.org/1.6/modules/generated/sklearn.neighbors.NearestNeighbors.html)
handles the indexing so you can focus on the target and training ideas.

**Think about it:** why could an incorrect confident anchor spread a mistake?


In [ ]:
# Use neighbouring training codes to share reliable cluster assignments.

@torch.no_grad()
def make_neighbour_targets(probabilities, neighbour_rows):
    """Spread confident cluster IDs through nearby AE codes; no digit labels are used."""
    values, cluster_ids = probabilities.topk(2, dim=1)
    # Compare the leading assignment with its closest alternative.
    confidence = values[:, 0] / values[:, 1]
    assignments = cluster_ids[:, 0]
    confident = torch.zeros(len(probabilities), dtype=torch.bool)

    # Keep the most confident quarter of each cluster as anchors.
    for cluster in range(probabilities.shape[1]):
        rows = torch.where(assignments == cluster)[0]
        ranked_rows = rows[torch.argsort(confidence[rows], descending=True)]
        confident[ranked_rows[:max(1, len(rows) // 4)]] = True

    # Anchor vectors represent predicted cluster IDs, not known digit labels.
    anchors = F.one_hot(assignments, probabilities.shape[1]).float()
    targets = probabilities.clone()
    for _ in range(40):
        # Average neighbouring rows to propagate local agreement.
        targets = targets[neighbour_rows].mean(dim=1)
        # Keep the reliable anchors fixed while the other rows are updated.
        targets[confident] = anchors[confident]
    return targets / targets.sum(dim=1, keepdim=True)


Start once from the fitted AE and the original mixture centres, then create Adam with learning rate `0.0001`.
Keep the five plotting keys `"batch"`, `"train"`, `"validation"`, `"reconstruction"` and `"extra"` in `dec_history`.

Reserve `min(20, DEC_EPOCHS // 3)` epochs for neighbour refinement. Let `balanced_epochs` be the remaining budget.
First use the supplied `mixture_targets` for `min(10, balanced_epochs // 3)` epochs. Then refresh balanced
targets from DEC each epoch. Switch to the neighbour phase when fewer than 0.1% of assignments change for three
consecutive epochs (after at least five epochs using DEC targets), or when the first phase exhausts its share of the budget.

At the switch, reuse `initial_neighbour_rows` from initialisation and call
`make_neighbour_targets(probabilities, neighbour_rows)`. Keep those targets fixed for the final phase.
Keep the learned model weights; start a fresh Adam and reset the loader generator to `SEED` for this phase.
The total number of completed epochs must stay within `DEC_EPOCHS`.

Before training, collect assignments, cluster sizes and code spread with `dec_training_status`.
After each epoch, record the losses, cluster sizes, coordinate spread and fraction of assignments
that changed. Use these measurements to assess whether the fit has become concentrated in too few
clusters. The supplied loop stops according to its phase and epoch budget; it does not reject
updates or restore an earlier model. No digit labels enter any training stage.


In [ ]:
# Refine one model through the target stages while keeping an epoch budget.

# Start from the pretrained AE and a fresh copy of the selected centres.
initial_centres = torch.tensor(cluster_initialiser.means_, dtype=torch.float32)
dec = DenseDEC(ae, initial_centres, code_mean, code_std)
dec_loader.generator.manual_seed(SEED)

# Update the copied encoder, decoder and cluster centres together.
# The original plain AE remains unchanged.
dec_optimiser = torch.optim.Adam(dec.parameters(), lr=0.0001)
dec_history = {
    "batch": [],
    "train": [],
    "validation": [],
    "reconstruction": [],
    "extra": [],
}

# Measure the initial cluster assignments and coordinate spread before any DEC updates.
probabilities, assignments, counts, spread = dec_training_status(dec, train_images)
# Record initial cluster sizes and coordinate spread for comparison after training.
dec_diagnostics = {
    "cluster_counts": [counts.tolist()],
    "code_spread": [spread.tolist()],
    "assignment_change": [0.0],
    "phase": ["initial"],
}
stable_epochs = 0
# Divide one overall epoch allowance among the target stages.
neighbour_epochs = min(20, DEC_EPOCHS // 3)
balanced_epochs = DEC_EPOCHS - neighbour_epochs
mixture_epochs = min(10, balanced_epochs // 3)
phase = "mixture" if mixture_epochs else "balanced"
neighbour_steps = 0
print("Initial cluster sizes:", counts.tolist())

In [ ]:
# DEC MODEL TRAINING: refine the AE's latent space to improve clustering without using digit labels.


# Loop over the DEC epochs, switching between target types and recording measurements.
for epoch in range(DEC_EPOCHS):
    # Switch to the next target type if the current phase has run its course.
    if phase == "mixture" and epoch >= mixture_epochs:
        phase = "balanced"
        print("Switching to refreshed DEC targets.")
    if phase == "mixture":
        # Begin with the density model's full probability assignments.
        targets = mixture_targets
    elif phase == "balanced":
        # Refresh self-generated targets only between complete epochs.
        targets = target_distribution(probabilities)
    

    ###########   MODEL TRAINING   ###########

    # Neighbour targets are computed once at the phase switch and then held fixed.
    batch_losses = train_dec_epoch(dec, dec_optimiser, targets)

    # Measure the current cluster sizes, code spread and assignment changes without using digit labels.
    new_probabilities, new_assignments, new_counts, new_spread = dec_training_status(dec, train_images)

    ###########   MODEL TRAINING   ###########


    ###########   TRAINING HISTORY   ###########

    # Measure how many images changed component since the preceding epoch.
    changed = (new_assignments != assignments).float().mean().item()
    dec_history["batch"] += batch_losses
    record_dec(dec, dec_history)

    # Record the current cluster sizes, code spread and assignment change for plotting.
    dec_diagnostics["cluster_counts"].append(new_counts.tolist())
    dec_diagnostics["code_spread"].append(new_spread.tolist())
    dec_diagnostics["assignment_change"].append(changed)
    dec_diagnostics["phase"].append(phase)

    ###########   TRAINING HISTORY   ###########


    ###########   TRAINING EXIT CRITERIA   ###########

    # Carry the current assignments forward to the next epoch.
    probabilities, assignments = new_probabilities, new_assignments

    # Count consecutive epochs with very few assignment changes, to detect convergence.
    stable_epochs = stable_epochs + 1 if changed < 0.001 else 0

    if epoch == 0 or (epoch + 1) % 5 == 0 or epoch + 1 == DEC_EPOCHS:
        print(f"DEC epoch {epoch + 1} ({phase}): validation objective = {dec_history['validation'][-1]:.4f}; "
              f"assignments changed = {100 * changed:.2f}%; "
              f"mean code spread = {float(new_spread.mean()):.3f}")
        print("  Cluster sizes:", new_counts.tolist())

    # Switch to neighbour targets if the current phase has run its course or the assignments have settled.
    if phase in ("mixture", "balanced"):
        settled = epoch >= mixture_epochs + 4 and stable_epochs >= 3
        if settled or epoch + 1 == balanced_epochs:
            # The neighbour targets are computed once at the phase switch and then held fixed.
            if neighbour_epochs == 0:
                break
            neighbour_rows = initial_neighbour_rows

            # This last target is fixed for the remaining refinement steps.
            targets = make_neighbour_targets(probabilities, neighbour_rows)
            phase = "neighbours"
            dec_optimiser = torch.optim.Adam(dec.parameters(), lr=0.0001)
            dec_loader.generator.manual_seed(SEED)
            print(f"Switching to {neighbour_epochs} epochs with fixed neighbour targets.")
    else:
        neighbour_steps += 1

        # Stop the refinement if the neighbour targets have been used for the allocated number of epochs.
        if neighbour_steps == neighbour_epochs:
            break

    ###########   TRAINING EXIT CRITERIA   ###########


print("Completed refinement epochs:", len(dec_history["train"]))
print("Final cluster sizes:", dec_diagnostics["cluster_counts"][-1])


**Explanation:**

Each execution starts a fresh DEC model from the fitted AE and the original mixture centres.
A short initial stage uses the mixture probabilities. The next stage refreshes balanced DEC
targets. Once assignments stabilise, neighbour propagation builds a fixed target for the final
stage. The copied model keeps its learned weights; a fresh Adam optimiser starts the neighbour
stage without the accumulated momentum from the earlier stages.

The history records every completed epoch. Diagnostics include the initial state and the phase
of each update. All three stages share the same maximum epoch budget. There is no automatic
rollback: inspect the cluster sizes and coordinate spread before interpreting the fit.

Batch losses use the current phase’s targets. The epoch curves measure a balanced self-assignment
objective on a fixed 10,000-image training sample and all validation images. These epoch measurements
are comparable across phases, but they need not match the batch objective used for an update.

Neither target construction nor the phase switch uses digit labels. Labels are used afterwards
to interpret the groups. Overlap in two plotted coordinates alone does not establish that clusters
are mixed in the full latent space.


In [ ]:
# Inspect the completed clustering updates and their reconstructions.

# Plot the history if refinement ran; otherwise the model is still a copy of the pretrained AE.
if dec_history["train"]:
    plot_training(dec_history, "DEC with reconstruction: BCE + 0.1 × cluster KL")
    summarise_training(dec_history, "DEC with reconstruction")
else:
    print("No refinement epochs were run; the copied pretrained AE weights are unchanged.")
show_examples(dec, validation_images[:6], "DEC: same validation images")


In [ ]:
# Compare the same validation images before and after refinement.

def plot_dec_change(validation_before, validation_after, labels, dimensions=(0, 1)):
    """Compare the same validation images before and after DEC, coloured by digit."""
    # Dimensions is a tuple of two integers, e.g. (0, 1) for the first two latent coordinates.
    x_dimension, y_dimension = dimensions

    # Convert the tensors to DataFrames for easier plotting and add the labels as a new column.
    before = pd.DataFrame(validation_before.numpy())
    after = pd.DataFrame(validation_after.numpy())
    before["label"] = labels.numpy()
    after["label"] = labels.numpy()

    # Create a figure with two subplots, one for before and one for after DEC, sharing the same axes.
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True, sharey=True)
    for class_label, class_name in enumerate(CLASS_NAMES):

        # Hint: use the DataFrame's boolean indexing to select only the rows with the current class_label, for both before and after.
        before_class = ## FINISH_ME ## # Select rows from the 'before' DataFrame where the 'label' column equals class_label
        after_class = after[after["label"] == class_label]
        colour = f"C{class_label}"

        # Hint: use axes[0].scatter() to plot the before_class points and
        # axes[1].scatter() to plot the after_class points, using the same colour and label for both.

        # Plot the data before DEC on the first subplot
        axes[0].scatter( ## FINISH_ME ##, ## FINISH_ME ##,
                        color=colour, s=3, alpha=0.4, label=class_name)

        # Plot the data after DEC on the second subplot
        axes[1].scatter( ## FINISH_ME ##, ## FINISH_ME ##,
                        color=colour, s=3, alpha=0.4, label=class_name)

    axes[0].set_title("Before DEC")
    axes[1].set_title("After DEC")
    axes[0].set_xlabel(f"Latent coordinate {x_dimension}")
    axes[1].set_xlabel(f"Latent coordinate {x_dimension}")
    axes[0].set_ylabel(f"Latent coordinate {y_dimension}")
    axes[1].legend(title="Digit", loc="upper left", bbox_to_anchor=(1.02, 1))
    fig.suptitle("DEC refinement: the same validation images before and after")
    fig.tight_layout()
    plt.show()


**Supplied view selection — two existing coordinates**

Different clusters can overlap when we discard most latent coordinates. Select the pair that best preserves
the full-space cluster assignments on **training codes only**. No digit labels or validation data enter this choice.
Every point is still shown at its actual coordinates, and the colours still represent the true digit labels.
Use this same pair in both before/after panels. Both show the same validation images,
with all ten digit categories coloured consistently.


In [ ]:
# Choose a view using training assignments, without consulting digit labels.

@torch.no_grad()
def choose_latent_dimensions(training_codes, centres):
    """Choose two existing coordinates that best retain the full-space cluster assignments."""
    # Full-space assignments provide a reference for judging a two-coordinate view.
    full_assignments = torch.cdist(training_codes, centres).argmin(dim=1)
    best_dimensions = (0, 1)
    best_agreement = -1.0
    for x in range(training_codes.shape[1]):
        for y in range(x + 1, training_codes.shape[1]):
            # Select existing coordinates; do not rotate or refit the representation.
            dimensions = [x, y]
            assignments = torch.cdist(
                training_codes[:, dimensions],
                centres[:, dimensions],
            ).argmin(dim=1)
            agreement = (assignments == full_assignments).float().mean().item()
            if agreement > best_agreement:
                best_dimensions = (x, y)
                best_agreement = agreement
    return best_dimensions, 100 * best_agreement


In [ ]:
# Keep the full vectors for analysis, then select two coordinates for display.

# The DEC model's latent space is the same size as the AE's, so the same code collection method can be used.
dec_codes = collect_code_sets(dec)

# Add digit labels to the DEC latent coordinates for plotting, then calculate the validation codes before DEC refinement.
dec_training_latent = ## FINISH_ME ##
dec_training_latent["label"] = ## FINISH_ME ##
dec_validation_latent = pd.DataFrame(dec_codes["Validation"].numpy())
dec_validation_latent["label"] = validation_frame["label"].to_numpy()

# Display the same validation images before and after DEC, coloured by digit, using the two coordinates that best retain the full-space assignments.
dec_dimensions, view_agreement = choose_latent_dimensions(dec_codes["Training"], dec.KL_centres.detach())
print("DEC plot coordinates:", dec_dimensions)
print(f"Training full-space assignments retained in this view: {view_agreement:.1f}%")

# Calculate the validation codes before DEC refinement, using the AE's mean and standard deviation to standardise them.
before_dec = (ae_codes["Validation"] - dec.code_mean) / dec.code_std
plot_dec_change(before_dec, dec_codes["Validation"], validation_labels, dimensions=dec_dimensions)

### 5.2 Decode ten latent-space centroids


Use K-means to summarise the **trained IDEC latent space** with ten centroids.
Fit it to all of the training vectors collected above, using all latent coordinates.

Each centroid is an average position in latent space. Pass it through the IDEC **decoder** to see
the image that this position represents. First undo the coordinate scaling used for clustering,
because the decoder was trained with the original AE coordinates.

This step only measures the fitted model; it does not update the encoder or decoder.
The centroid numbers are arbitrary cluster IDs, not digit labels. This new K-means fit
summarises the refined codes; its cluster IDs need not match the learned IDEC centre IDs.


Decoding Centroids:

1. Fit K-means with ten clusters to the training latent vectors.
2. Retrieve its ten centre vectors and convert them to a floating-point tensor.
3. Undo the latent scaling, decode the vectors and display the ten images.


In [ ]:
# Fit ten representative points to the trained IDEC model's latent vectors.
from sklearn.cluster import KMeans

# Construct a KMeans object with 10 clusters, 10 initialisations, and a fixed random seed for reproducibility.
kmeans = KMeans(n_clusters=10, n_init=10, random_state=SEED)
# dec_codes is a dictionary; dec_codes["Training"] is the tensor of training latent vectors.
# We need to convert it to a NumPy array before fitting KMeans.
kmeans.fit( ## FINISH_ME ##)

# The centroids are the representative points in the DEC latent space, which we will decode into images.
centroids = torch.tensor(kmeans.cluster_centers_, dtype=torch.float32)

# Undo the latent-space scaling: the decoder expects the original AE coordinates.
raw_centroids = centroids * dec.code_std + dec.code_mean

# Decode each representative point into a 28 × 28 image.
dec.eval()
with torch.no_grad():
    # dec.dec_ae.decoder is the decoder updated during IDEC refinement.
    # It turns unscaled latent codes into reconstructed images.
    centroid_images = ## FINISH_ME ## # Pass the raw_centroids through the decoder to get the reconstructed images.

# Display one reconstructed image for each K-means centroid.
fig, axes = plt.subplots(1, 10, figsize=(15, 2.5))

# Loop over each centroid and display its reconstructed image in the corresponding subplot.
for index in range(10):
    # Hint: use axes[index].imshow() to display the centroid image, and set the title and axis properties.
    axes[index].imshow(centroid_images[index, 0], cmap="gray", vmin=0, vmax=1)
    axes[index].set_title(f"Centroid {index}")
    axes[index].axis("off")

fig.tight_layout()
plt.show()


**Explanation:**

K-means returns one representative vector per cluster. Rescaling puts those vectors back into
the coordinate system expected by the trained decoder. The ten output images show what the
decoder associates with those representative positions; they are not selected input images.


### 5.3 Which digit is most common around each centroid?


Use the **same K-means fit from Section 5.2**. Its stored assignments tell us which
training images belong to each centroid. Join those assignments to the images' known digit labels,
keeping their row order unchanged.

For each centroid:

1. Count how often each digit label occurs among its assigned images.
2. Pick the label with the largest count.
3. Divide that count by the number of images in the cluster and multiply by 100.

Compare the resulting table with the ten decoded images above. This is a description of the
fitted clusters; it does not use labels to retrain the model. Two centroids may have the same most
common digit. The percentage shows how strongly one digit dominates each group.


In [ ]:
# Pair each K-means assignment with the corresponding training image's digit label.
centroid_members = pd.DataFrame({
    "centroid": kmeans.labels_,
    "digit": train_frame["label"].to_numpy(),  # Keep the same image order as the latent vectors.
})

summary_rows = []
# Loop over each centroid to summarise the digit distribution among its assigned images.
for index in range(10):
    # Select only the digit labels of images assigned to this centroid.
    cluster_digits = centroid_members.loc[centroid_members["centroid"] == index, "digit"]

    # Count how many images of each digit are assigned to this centroid.
    counts = cluster_digits.value_counts()

    # The most frequent label describes the largest group around this centroid.
    # Hint: use counts.idxmax() to find the most common digit and counts.max() to find its count.
    most_common_digit = ## FINISH_ME ##
    # Hint: use counts.max() to find the count of the most common digit.
    most_common_count = ## FINISH_ME ##

    # Calculate the share of this most common digit among all images assigned to this centroid.
    share = 100 * most_common_count / len(cluster_digits)

    # Append a summary row for this centroid, including its index, most common digit, count,
    # total images, and share percentage.
    summary_rows.append([index, most_common_digit, most_common_count, len(cluster_digits), round(share, 1)])

centroid_summary = pd.DataFrame(summary_rows, columns=["Centroid", "Most common digit",
                                                       "Count for this digit", "Total images", "Share (%)"])
display(centroid_summary)


**Question: do Sections 5.2 and 5.3 match?**

For each centroid, compare its decoded image with the most common digit in the table.
Identify any mismatches and use the percentage share to discuss whether that centroid
represents one clear digit or a mixture.


**Answer:**

`## FINISH_ME ##`


## 6.0 Anomaly investigation — the plain dense autoencoder


This investigation uses **only the plain dense AE trained in Section 2**, with its weights fixed.
Your question is whether a model trained on digits treats a smiley face as unusual.

Before starting:

1. Set the fitted AE to evaluation mode and perform measurements without recording gradients. Do not retrain it.
2. Load **smiley_28x28_inverted.png** as a single 28 × 28 greyscale image. Apply the same intensity scale and
   foreground/background convention as the digit images. Display it and check the orientation by eye.
3. Divide the held-out validation digits into two equal groups. Use one group for **calibration** and the other
   for **evaluation**. Keep their image labels aligned. Keep the official test set aside.
4. Use the calibration digits to choose histogram bins and the **99th-percentile reference cut** for each scalar
   score below. Keep those choices fixed when examining evaluation digits and the smiley.
5. Use training digits only when estimating latent-space reference quantities, such as class means and coordinate scales.

Think of this as a measurement problem: define an observable, measure its reference distribution,
locate the unfamiliar image within that distribution, and assess what the evidence supports.


### 6.1 (i) Where does the smiley fall in the per-image loss distributions?


Let $x_{ip}$ be pixel $p$ of image $i$, and let $\hat{x}_{ip}$ be its AE reconstruction.
There are $P=784$ pixels per image. Define the residual and squared residual by

$$
r_{ip}=\hat{x}_{ip}-x_{ip}, \qquad E_{ip}=r_{ip}^{\,2}.
$$

Measure two scalar observables for each image:

$$
S_i^{\mathrm{MSE}}=\frac{1}{P}\sum_{p=1}^{P}E_{ip},
$$

$$
S_i^{\mathrm{BCE}}=-\frac{1}{P}\sum_{p=1}^{P}
\left[x_{ip}\log\hat{x}_{ip}+(1-x_{ip})\log(1-\hat{x}_{ip})\right].
$$

Both definitions average over pixels and leave **one number per image**. The mean BCE is the
reconstruction quantity used in Section 2. A sum over pixels differs by the constant factor 784;
keep the chosen convention consistent and label the axes accordingly.

Work through the following measurements:

1. Reconstruct the calibration and evaluation digits with the fixed AE. Retain one MSE and one mean BCE per image.
2. Reconstruct the smiley and calculate the same two scores.
3. Make one figure per score. Show the normal-digit reference distribution and distinguish the evaluation distribution.
   Mark the smiley's score and the calibration cut with clearly labelled vertical lines.
4. Report the smiley's percentile in the calibration distribution. For a reference sample of size $N$, use

$$
\text{percentile}(s)=\frac{100}{N}\sum_{i=1}^{N}\mathbf{1}(S_i\leq s).
$$

**Record your findings:** Where is the smiley relative to the bulk and tail of each distribution?
Do MSE and BCE lead to the same ordering? Is a percentile a probability that an image is anomalous?
Support your interpretation with your measured values rather than the appearance of the smiley.


### 6.2 (ii) What do the per-class, per-pixel error distributions look like?


The previous step assigned one score to each image. Now examine the individual squared residuals
$E_{ip}$, before averaging over pixels.

1. Use the digit labels in your Pandas table to select each class in turn.
2. For each class, reconstruct the corresponding evaluation images with the same plain AE.
3. Pool the squared residuals from **every pixel of every selected image**. A class containing $N_c$ images
   contributes $784N_c$ values to its distribution.
4. Choose bin edges from the calibration digits’ pixel errors, then keep them fixed.
   Produce ten labelled panels for the evaluation digits, one per class, with common bins and axis limits.
   Use probability density or fractions of pixels so differences in class sample size do not dominate.
5. Choose a display scale that reveals both the bulk and the tail. If you use a logarithmic scale,
   state how exact zeros are displayed; leave the underlying residuals unchanged.

These are distributions of **pixel errors**, not distributions of image means and not pictures of the
mean error at each pixel location. Keep those three objects distinct.

**Record your findings:** Which classes have broader tails? Do many small residuals coexist with a small
number of large ones? Could a difference between digit classes affect a single anomaly threshold?
Explain what information disappears when the pixel errors are replaced by one image average.


### 6.3 (iii) What do the smiley's pixel distributions and error map show?


Examine the smiley itself at pixel level. Keep **pixel intensity** and **reconstruction error**
as separate observables.

Produce the following evidence:

1. Show the input smiley, its AE reconstruction and a map of its squared residuals.
   Label the image panels and include an error-scale colour bar.
2. Plot the distribution of the smiley's **784 input intensities**. You may overlay the reconstructed
   intensities to examine how their distributions differ.
3. Plot the distribution of its **784 squared residuals** using the same binning and scale as the
   digit-class error distributions in Section 6.2. Make a direct comparison with the normal-digit reference.
4. Relate features in the histogram to locations in the error map. Identify which parts of the image
   contribute most to the total squared residual.
5. Compare brighter input pixels with background pixels. State how you distinguish the two groups
   before interpreting their errors, and record how many pixels belong to each.

A histogram records how often values occur; an error map records where they occur.
Use both forms of evidence.

**Record your findings:** Does the reconstruction resemble the input in the places that matter?
How do the relative amounts of foreground and background influence an image-average score?
Does the smiley's pixel-error distribution differ from every digit class, or overlap with some of them?


### 6.4 (iv) Where is the smiley in the AE latent space?


Use the **plain AE encoder** and the training and validation representations introduced in Section 4.
All reference vectors and the smiley's vector must come from this same fitted model.

Start with the visual investigation:

1. Encode the smiley once to obtain its full latent vector.
2. Choose two existing latent coordinates using the training digits, before examining the smiley's position.
3. Show the training and evaluation digit clouds in side-by-side panels, with all ten classes coloured consistently.
   Add the smiley as a clearly distinguishable marker, using the same coordinates and axis scales in both panels.
4. Inspect at least one other coordinate pair. Record whether your visual interpretation changes.

Then make a quantitative measurement using **all** latent coordinates. Estimate a reference mean $m_c$
for each digit class from its training vectors. Let $s_j$ be the standard deviation of coordinate $j$
over the training vectors. Use the dimensionless distance

$$
D(z)=\min_c \sqrt{\sum_{j=1}^{d}
\left(\frac{z_j-m_{cj}}{s_j}\right)^2}.
$$

The coordinate scales and class means are fixed by training digits; do not re-estimate them using the smiley.
If a coordinate is constant across all training images, leave it out of this distance.

5. Calculate this distance for the calibration digits, the evaluation digits and the smiley.
6. Plot its normal-digit distributions, with the smiley's distance and the calibration cut marked.
7. Report the smiley's reference percentile and compare it with the reconstruction-score percentiles.

**Record your findings:** Does the two-coordinate view agree with the full-dimensional distance?
Why can two projected clouds overlap even when their full vectors differ?
Does being near a class mean demonstrate that an image is a valid member of that class?


### 6.5 Combine the evidence and test your decision rule


For each of the three scalar scores—image MSE, mean image BCE and full-dimensional latent distance—
use the calibration cut fixed earlier. Count how many evaluation digits lie above it:

$$
\text{false-alarm fraction}=
\frac{\text{evaluation digits above the cut}}{\text{number of evaluation digits}}.
$$

Apply the same rule to the smiley. Keep the model, score definitions, reference samples and cuts fixed
throughout this comparison.

Complete an evidence table with these columns:

| Score | Smiley value | Calibration percentile | Calibration cut | Evaluation-digit false-alarm fraction | Smiley above the cut? |
|---|---|---|---|---|---|
| Image MSE | | | | | |
| Mean image BCE | | | | | |
| Latent distance | | | | | |

Write a short conclusion that addresses all four parts of the investigation:

- Where did the smiley fall in the per-image loss distributions?
- What did the class-dependent pixel errors add to that interpretation?
- What did its intensity histogram and spatial error map reveal?
- Did latent-space evidence identify the anomaly consistently?

State which claim your measurements support, which remains uncertain, and what additional normal and
anomalous examples you would need for a stronger test. Consider whether one smiley can establish a
general detection efficiency, or whether a fixed false-alarm fraction establishes an anomaly probability.

Your conclusion should cite measured values and your own figures. There is no supplied numerical
answer or model ranking for this investigation.
